In [ ]:
# pip install astroquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 40.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.6/997.6 kB 43.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
from astroquery.gaia import Gaia
import pandas as pd

In [2]:
Gaia.login(user="atiwar04", password="%u69kDebbie")

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


In [3]:
tables = Gaia.load_tables(only_names=True)
table_names = [t.get_qualified_name() for t in tables]
print(table_names)

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
['external.apassdr9', 'external.catwise2020', 'external.gaiadr2_astrophysical_parameters', 'external.gaiadr2_geometric_distance', 'external.gaiaedr3_distance', 'external.gaiaedr3_gcns_main_1', 'external.gaiaedr3_gcns_rejected_1', 'external.gaiaedr3_spurious', 'external.gaia_eso_survey', 'external.galex_ais', 'external.lamost_dr9_lrs', 'external.lamost_dr9_mrs', 'external.ravedr5_com', 'external.ravedr5_dr5', 'external.ravedr5_gra', 'external.ravedr5_on', 'external.ravedr6', 'external.sdssdr13_photoprimary', 'external.skymapperdr1_master', 'external.skymapperdr2_master', 'external.tmass_xsc', 'external.xgboost_table1', 'external.xgboost_table2', 'gaiadr1.aux_qso_icrf2_match', 'gaiadr1.ext_phot_zero_point', 'gaiadr1.allwise_best_neighbour', 'gaiadr1.allwise_neighbourhood', 'gaiadr1.gsc23_best_neighbour', 'gaiadr1.gsc23_neighbourhood', 'gaiadr1

In [ ]:
# working
query = """
SELECT TOP 100000  -- Start smaller
    ra, dec, parallax, parallax_error, 
    phot_g_mean_mag, phot_bp_mean_mag, 
    phot_rp_mean_mag, 
    bp_rp, radial_velocity, ruwe, 
    teff_gspphot, phot_variable_flag
FROM gaiadr3.gaia_source
WHERE 
    parallax > 10
    AND bp_rp IS NOT NULL
    AND phot_g_mean_mag < 15
    AND ruwe < 1.4
    AND teff_gspphot IS NOT NULL
"""



| Column               | Description                                                                 | Project Relevance                                                                 |
|----------------------|-----------------------------------------------------------------------------|-----------------------------------------------------------------------------------|
| `ra`, `dec`          | Right Ascension and Declination (sky coordinates in degrees).              | Spatial mapping of stars for cluster identification.                              |
| `parallax`           | Parallax (milliarcseconds). Convert to distance: `distance_pc = 1000/parallax`. | Calculate distances to stars. Critical for luminosity calculations.               |
| `parallax_error`     | Uncertainty in parallax measurement.                                       | Filter stars with reliable distances (`parallax_error/parallax < 0.2`).           |
| `phot_g_mean_mag`    | Apparent magnitude in Gaia’s broad "G" band.                               | Calculate absolute magnitude (`M_G = G - 5 log10(distance) + 5`).                 |
| `phot_bp_mean_mag`   | Blue Photometer (BP) magnitude.                                            | Stellar temperature/color analysis.                                               |
| `phot_rp_mean_mag`   | Red Photometer (RP) magnitude.                                             | Stellar temperature/color analysis.                                               |
| `bp_rp`              | BP - RP color index.                                                       | Proxy for temperature (redder = cooler, bluer = hotter).                          |
| `radial_velocity`    | Line-of-sight velocity (km/s).                                             | Kinematic studies (e.g., Galactic structure, cluster membership).                 |
| `ruwe`               | Renormalized Unit Weight Error (astrometric quality flag).                | Filter high-quality data (`ruwe < 1.4` = reliable positions).                     |
| `teff_gspphot`       | Effective temperature (Kelvin) from Gaia Photometry.                       | Direct input for Hertzsprung-Russell (HR) diagrams and stellar evolution modeling.|
| `phot_variable_flag` | Flag indicating stellar variability.                                       | Identify pulsating stars (e.g., Cepheids, RR Lyrae) for phase classification.     |

In [19]:
job = Gaia.launch_job_async(query)
df = job.get_results().to_pandas()

INFO: Query finished. [astroquery.utils.tap.core]


In [20]:
df.to_csv("gaia_stars.csv", index=False)

In [21]:
data= pd.read_csv('gaia_stars.csv')

In [ ]:
data.head()

,ra,dec,parallax,parallax_error,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp,radial_velocity,ruwe,teff_gspphot,phot_variable_flag
0,251.589977,-50.638635,13.398257,0.017850,13.470652,14.585335,12.419784,2.165551,12.315269,0.951887,3730.103,NOT_AVAILABLE
1,250.185079,-51.477674,19.588654,0.025484,6.224640,6.379528,5.939466,0.440062,45.425500,0.782458,7051.192,VARIABLE
2,250.772902,-51.598985,10.326266,0.020964,13.695807,14.826818,12.635778,2.191039,7.051061,1.033412,3796.808,NOT_AVAILABLE
3,317.395254,67.908576,11.136129,0.017837,14.756236,16.031088,13.637453,2.393635,-40.089650,1.133741,3564.809,NOT_AVAILABLE
4,295.130352,70.286247,17.357228,0.010188,10.476051,11.063625,9.755102,1.308523,1.091031,1.091369,4607.393,VARIABLE


In [23]:
data.tail()

,ra,dec,parallax,parallax_error,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp,radial_velocity,ruwe,teff_gspphot,phot_variable_flag
73143,324.935474,22.240966,10.884371,0.017102,13.748566,14.487608,12.913438,1.574170,-139.151400,1.043562,3995.3284,NOT_AVAILABLE
73144,256.116435,-28.583872,54.725979,0.023558,6.380115,6.777419,5.814102,0.963317,14.082294,0.739388,5421.7110,NOT_AVAILABLE
73145,256.498469,-28.887167,14.366089,0.017620,12.384461,13.256581,11.459989,1.796593,23.231743,0.742103,4022.2212,NOT_AVAILABLE
73146,256.843330,-28.319041,18.965919,0.020215,9.989574,10.567817,9.272490,1.295327,-12.614517,0.862961,4592.8467,VARIABLE
73147,264.506810,-38.750636,10.811695,0.031132,14.805878,16.222614,13.626999,2.595615,-67.894700,1.033955,3499.6277,NOT_AVAILABLE


---

##### Test



In [7]:
query_g = """
SELECT TOP 10000  -- Start small to avoid server errors
    g.ra, g.dec, g.parallax, g.parallax_error, 
    g.phot_g_mean_mag, g.phot_bp_mean_mag, g.phot_rp_mean_mag, 
    g.bp_rp, g.radial_velocity, g.ruwe, 
    ap.teff_gspphot, ap.luminosity_gspphot, ap.radius_gspphot, ap.feh_gspphot
FROM gaiadr3.gaia_source AS g
JOIN gaiadr3.astrophysical_parameters AS ap 
    ON g.source_id = ap.source_id
WHERE 
    g.parallax > 10
    AND g.bp_rp IS NOT NULL
    AND g.phot_g_mean_mag < 15
    AND g.ruwe < 1.4
    AND ap.teff_gspphot IS NOT NULL
    AND ap.luminosity_gspphot IS NOT NULL
    AND ap.feh_gspphot IS NOT NULL
"""

In [8]:
job_g = Gaia.launch_job_async(query_g)
df_g = job_g.get_results().to_pandas()

500 Error 500:
null


HTTPError: Error 500:
null

In [5]:
# Query astrophysical_parameters
query_ap = """
SELECT TOP 1000
    source_id, teff_gspphot, luminosity_gspphot, feh_gspphot
FROM gaiadr3.astrophysical_parameters
WHERE
    teff_gspphot IS NOT NULL
    AND luminosity_gspphot IS NOT NULL
    AND feh_gspphot IS NOT NULL
"""

In [6]:
job_ap = Gaia.launch_job_async(query_ap)
df_ap = job_ap.get_results().to_pandas()


500 Error 500:
null


HTTPError: Error 500:
null

##### Probable Error
The HTTP 500 error persists because Gaia’s servers struggle with complex queries on the astrophysical_parameters table for unregistered users or due to server-side constraints. Let’s refine your working query and integrate additional parameters incrementally

##### Error
query = """
- SELECT TOP 10000 <br>
    g.ra, <br>
    g.dec, <br>
    g.parallax,<br> 
    g.parallax_error, <br>
    g.phot_g_mean_mag, <br>
    g.phot_bp_mean_mag, <br>
    g.phot_rp_mean_mag, <br>
    g.bp_rp, <br>
    g.radial_velocity, <br>
    g.ruwe, <br>
    g.phot_variable_flag,<br>
    ap.teff_gspphot,      -- From astrophysical_parameters<br>
    ap.luminosity_gspphot,<br>
    ap.feh_gspphot<br>
- FROM gaiadr3.gaia_source AS g<br>
- JOIN gaiadr3.astrophysical_parameters AS ap <br>
    ON g.source_id = ap.source_id  -- Join using source_id<br>
- WHERE <br>
    g.parallax > 10 <br>
    AND g.bp_rp IS NOT NULL<br>
    AND g.phot_g_mean_mag < 15<br>
    AND g.ruwe < 1.4<br>
"""